In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict


In [4]:
class batsmanState(TypedDict):
    runs : int
    balls : int
    four : int
    six : int
    
    sr: float #run rate
    bpb : float #boundary per ball
    boundary_percent : float
    summary : str

In [17]:
def calc_sr(state:batsmanState)->batsmanState:
    sr = (state['runs']/state['balls'])*100
    state['sr']=sr
    return {'sr': sr}

def calc_bpb(state:batsmanState)->batsmanState:
    bpb = state['balls']/(state['four'] + state['six'])
    state['bpb']=bpb
    return {'bpb': bpb}

def calc_boundary_percent(state: batsmanState):
    boundary_percent = (((state['four'] * 4) + (state['six'] * 6))/state['runs'])*100
    state['boundary_percent']=boundary_percent
    return {'boundary_percent': boundary_percent}

def summary(state: batsmanState):
    summary = f"""
Strike Rate - {state['sr']} \n
Balls per boundary - {state['bpb']} \n
Boundary percent - {state['boundary_percent']}
"""
    state['summary'] = summary
    return {'summary': summary}


In [18]:
graph = StateGraph(batsmanState)

graph.add_node('calc_sr',calc_sr)
graph.add_node('calc_bpb',calc_bpb)
graph.add_node('calc_boundary_percent', calc_boundary_percent)
graph.add_node('summary', summary)


In [19]:
graph.add_edge(START, 'calc_sr')
graph.add_edge(START, 'calc_bpb')
graph.add_edge(START, 'calc_boundary_percent')

graph.add_edge('calc_sr', 'summary')
graph.add_edge('calc_bpb', 'summary')
graph.add_edge('calc_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [20]:
intial_state = {
    'runs': 100,
    'balls': 50,
    'four': 6,
    'six': 4
}

final_state = workflow.invoke(intial_state)
print(final_state)

{'runs': 100, 'balls': 50, 'four': 6, 'six': 4, 'sr': 200.0, 'bpb': 5.0, 'boundary_percent': 48.0, 'summary': '\nStrike Rate - 200.0 \n\nBalls per boundary - 5.0 \n\nBoundary percent - 48.0\n'}
